Dataset Overview: DAIC-WOZ (FULL 189 PARTICIPANTS)
**Pipeline v49** — RICHEST FEATURE ENGINEERING: BERT + COVAREP PROSODIC + LINGUISTIC MARKERS

─────────────────────────────────────────────────────────────────────
 v49 = 189 Partisipan + Rich Features + 5-Fold StratifiedCV + Bayesian Tuning

 Analisis v48: COVAREP temporal features kurang informatif. SVM_rbf CV AUC=0.63
 menunjukkan audio features ada sinyal tapi terlalu noisy. Test set 38 terlalu kecil.

 Strategi Baru:
 [1] BERT Embeddings (v13, 384 fitur) — terbukti paling kuat
 [2] COVAREP Prosodic Features (fitur kunci depresi: F0/pitch, energy, jitter, shimmer)
     — ekstrak hanya 12 fitur yang PALING RELEVAN secara klinis
 [3] Linguistic Richness Features dari transcript (word count, pauses, response length)
     — fitur yang belum pernah digunakan
 [4] 5-Fold StratifiedCV (bukan 80/20) untuk evaluasi yang lebih stabil
 [5] Calibrated SVM + LR + Gradient Boosting dengan Optuna/RandomizedSearchCV
 [6] Label-weighted ensemble dengan temperature scaling
─────────────────────────────────────────────────────────────────────


## Setup & Imports


In [ ]:
import os, sys, glob, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
import re

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, roc_auc_score, classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV

from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTETomek

import xgboost as xgb
import lightgbm as lgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if "notebooks" in os.getcwd() else os.getcwd()
RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "DAIC-WOZ")
V13_DIR = os.path.join(PROJECT_ROOT, "data", "features", "v13")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v49")

for d in [os.path.join(RESULTS_DIR, "metrics")]:
    os.makedirs(d, exist_ok=True)


## Load Labels (189 Participants)


In [ ]:
df_train_raw = pd.read_csv(os.path.join(RAW_DIR, "train_split_Depression_AVEC2017.csv"))
df_dev_raw   = pd.read_csv(os.path.join(RAW_DIR, "dev_split_Depression_AVEC2017.csv"))
df_test_raw  = pd.read_csv(os.path.join(RAW_DIR, "full_test_split.csv"))

df_train_raw = df_train_raw[['Participant_ID', 'PHQ8_Binary', 'Gender']].rename(
    columns={'Participant_ID': 'id', 'PHQ8_Binary': 'label', 'Gender': 'gender'})
df_dev_raw = df_dev_raw[['Participant_ID', 'PHQ8_Binary', 'Gender']].rename(
    columns={'Participant_ID': 'id', 'PHQ8_Binary': 'label', 'Gender': 'gender'})
df_test_raw = df_test_raw[['Participant_ID', 'PHQ_Binary', 'Gender']].rename(
    columns={'Participant_ID': 'id', 'PHQ_Binary': 'label', 'Gender': 'gender'})

df_labels = pd.concat([df_train_raw, df_dev_raw, df_test_raw], ignore_index=True)
df_labels['id'] = df_labels['id'].astype(int)
df_labels = df_labels.reset_index(drop=True)

print(f"Total participants: {len(df_labels)}")
print(f"Label distribution: {df_labels['label'].value_counts().to_dict()}")
print(f"Gender distribution: {df_labels['gender'].value_counts().to_dict()}")


## Feature 1: BERT Text Embeddings (384 fitur)


In [ ]:
print("\n=== Loading BERT Embeddings (v13) ===")
df_bert = pd.read_csv(os.path.join(V13_DIR, "v13_text_embeddings.csv"))
bert_cols = [c for c in df_bert.columns if c.startswith('text_emb_')]
df_labels = df_labels.merge(df_bert[['participant_id'] + bert_cols],
                             left_on='id', right_on='participant_id', how='left')
X_bert = df_labels[bert_cols].values.astype(np.float64)
X_bert = np.nan_to_num(X_bert, nan=0.0)
print(f"BERT shape: {X_bert.shape}")


## Feature 2: Clinically-Relevant COVAREP Prosodic Features


In [ ]:
print("\n=== Extracting Clinically-Relevant Prosodic Features ===")
# Columns based on COVAREP documentation:
# Col 0: F0 (pitch in Hz), Col 1: Voiced prob, Col 2: NAQ, Col 3: QOQ,
# Col 4: H1H2, Col 5: PSP, Col 6: MDQ, Col 7: peakSlope,
# Col 8: Rd, Col 9: Rd_conf, Cols 10-21: MCEP (mel-cepstral coefficients),
# Col 22: HMPDM_0 ... Cols 23-64: more spectral features
# Col 65: VUV, Cols 66-73: MFCC-like

# Clinical markers of depression in voice:
# - Low/reduced pitch variability (monotone voice)
# - Reduced speech rate, longer pauses
# - Changes in voice quality (jitter, shimmer via NAQ, QOQ)
# - Reduced energy/loudness

def extract_prosodic_features(pid, raw_dir):
    """Extract clinically-relevant prosodic features for depression detection."""
    filepath = os.path.join(raw_dir, f"{pid}_P", f"{pid}_COVAREP.csv")
    if not os.path.exists(filepath):
        return np.zeros(30)

    try:
        df = pd.read_csv(filepath, header=None)
        data = df.values.astype(np.float64)
        data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

        n_frames = len(data)
        voiced_mask = data[:, 1] > 0.5
        n_voiced = voiced_mask.sum()
        voiced_ratio = n_voiced / max(n_frames, 1)

        # F0 (pitch) features — key depression marker
        f0_all = data[:, 0]
        f0_voiced = f0_all[voiced_mask] if n_voiced > 5 else f0_all[f0_all > 0]
        if len(f0_voiced) > 5:
            f0_mean   = np.mean(f0_voiced)
            f0_std    = np.std(f0_voiced)  # monotone = low std
            f0_range  = np.ptp(f0_voiced)  # pitch range
            f0_cv     = f0_std / (f0_mean + 1e-8)  # coefficient of variation
            # Semi-tone variation (perceptual pitch variability)
            f0_semitone_std = np.std(12 * np.log2(f0_voiced / 110 + 1e-8))
        else:
            f0_mean = f0_std = f0_range = f0_cv = f0_semitone_std = 0.0

        # NAQ (Normalized Amplitude Quotient) — voice quality
        naq = data[:, 2]
        naq_voiced = naq[voiced_mask] if n_voiced > 5 else naq
        naq_mean = np.mean(naq_voiced) if len(naq_voiced) > 0 else 0
        naq_std  = np.std(naq_voiced)  if len(naq_voiced) > 0 else 0

        # QOQ — glottal opening quotient
        qoq = data[:, 3]
        qoq_voiced = qoq[voiced_mask] if n_voiced > 5 else qoq
        qoq_mean = np.mean(qoq_voiced) if len(qoq_voiced) > 0 else 0

        # Energy / spectral features (cols 6-9)
        energy_cols = data[:, 6:10]
        energy_mean = np.mean(energy_cols, axis=0)  # 4 values
        energy_std  = np.std(energy_cols, axis=0)   # 4 values

        # Pauses analysis (silence = not voiced frames in a row)
        silence_runs = []
        in_silence = False
        run_len = 0
        for v in voiced_mask:
            if not v:
                in_silence = True
                run_len += 1
            else:
                if in_silence and run_len > 3:  # >3 frames = real pause
                    silence_runs.append(run_len)
                in_silence = False
                run_len = 0
        n_pauses = len(silence_runs)
        mean_pause_dur = np.mean(silence_runs) if silence_runs else 0
        max_pause_dur  = np.max(silence_runs) if silence_runs else 0

        # MFCC-like first coefficient (brightness/spectral centroid proxy)
        if data.shape[1] > 22:
            mcep0 = data[:, 22]
            mcep0_voiced = mcep0[voiced_mask] if n_voiced > 5 else mcep0
            mcep_mean = np.mean(mcep0_voiced) if len(mcep0_voiced) > 0 else 0
            mcep_std  = np.std(mcep0_voiced)  if len(mcep0_voiced) > 0 else 0
        else:
            mcep_mean = mcep_std = 0.0

        # Combine all features
        features = np.array([
            voiced_ratio,        # Speech activity
            f0_mean,             # Average pitch
            f0_std,              # Pitch variability (monotone = low)
            f0_range,            # Pitch range
            f0_cv,               # Pitch coefficient of variation
            f0_semitone_std,     # Perceptual pitch variability
            naq_mean,            # Voice quality
            naq_std,
            qoq_mean,
            *energy_mean,        # 4 energy features
            *energy_std,         # 4 energy features
            n_pauses / max(n_frames/100, 1),   # Pause rate
            mean_pause_dur,      # Average pause duration
            max_pause_dur,       # Longest pause
            mcep_mean,           # Spectral brightness
            mcep_std,
            n_voiced / max(n_frames, 1),  # Voiced rate
        ])
        return features[:30]  # Ensure exactly 30 features
    except Exception as e:
        return np.zeros(30)

prosodic_feats = []
for _, row in df_labels.iterrows():
    pid = int(row['id'])
    feats = extract_prosodic_features(pid, RAW_DIR)
    prosodic_feats.append(feats)

X_prosodic = np.array(prosodic_feats)
X_prosodic = np.nan_to_num(X_prosodic, nan=0.0, posinf=0.0, neginf=0.0)
print(f"Prosodic features shape: {X_prosodic.shape}")


## Feature 3: Linguistic & Behavioral Features from Transcripts


In [ ]:
print("\n=== Extracting Linguistic Features from Transcripts ===")

def extract_linguistic_features(pid, raw_dir):
    """Extract linguistic markers of depression from transcripts."""
    filepath = os.path.join(raw_dir, f"{pid}_P", f"{pid}_TRANSCRIPT.csv")
    if not os.path.exists(filepath):
        return np.zeros(20)

    try:
        df_trans = pd.read_csv(filepath, sep='\t')

        # Separate participant and Ellie turns
        if 'speaker' in df_trans.columns:
            df_part = df_trans[df_trans['speaker'].str.lower() == 'participant'].copy()
            df_ellie = df_trans[df_trans['speaker'].str.lower() == 'ellie'].copy()
        else:
            df_part = df_trans.copy()
            df_ellie = pd.DataFrame()

        if 'value' not in df_part.columns or len(df_part) == 0:
            return np.zeros(20)

        # Participant text
        participant_texts = df_part['value'].dropna().astype(str).tolist()
        all_text = " ".join(participant_texts).lower()

        # Basic counts
        n_turns = len(df_part)
        words = all_text.split()
        n_words = len(words)
        unique_words = len(set(words))
        lexical_diversity = unique_words / max(n_words, 1)  # Type-Token Ratio
        avg_words_per_turn = n_words / max(n_turns, 1)

        # Depression-related word categories
        first_person_words = ['i', "i'm", "i've", "i'll", 'my', 'me', 'myself', 'mine']
        negative_affect_words = ['sad', 'depressed', 'tired', 'exhausted', 'hopeless',
                                  'worthless', 'fail', 'alone', 'lonely', 'empty',
                                  'anxious', 'worried', 'stress', 'bad', 'worse', 'worst',
                                  'never', 'nothing', 'nobody', 'no one', "can't", 'cannot']
        positive_words = ['happy', 'good', 'great', 'fine', 'well', 'okay', 'enjoy',
                           'love', 'nice', 'wonderful', 'excellent', 'better', 'best']
        filler_words = ['um', 'uh', 'like', 'you know', 'kind of', 'sort of', 'i mean']

        word_count = len(words)
        first_person_rate  = sum(words.count(w) for w in first_person_words) / max(word_count, 1)
        negative_rate      = sum(all_text.count(w) for w in negative_affect_words) / max(word_count, 1)
        positive_rate      = sum(all_text.count(w) for w in positive_words) / max(word_count, 1)
        filler_rate        = sum(all_text.count(w) for w in filler_words) / max(word_count, 1)
        sentiment_ratio    = positive_rate / max(negative_rate + 1e-8, 1e-8)

        # Sentence-level features
        sentences = re.split(r'[.!?]+', all_text)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
        avg_sent_length = np.mean([len(s.split()) for s in sentences]) if sentences else 0
        sent_length_var = np.std([len(s.split()) for s in sentences]) if len(sentences) > 1 else 0

        # Response timing (from timestamps)
        response_latencies = []
        if 'start_time' in df_trans.columns and 'stop_time' in df_trans.columns:
            turns = df_trans.sort_values('start_time').reset_index(drop=True)
            for i in range(1, len(turns)):
                if (turns.iloc[i]['speaker'].lower() == 'participant' and
                    turns.iloc[i-1]['speaker'].lower() == 'ellie'):
                    latency = turns.iloc[i]['start_time'] - turns.iloc[i-1]['stop_time']
                    if 0 < latency < 30:  # Valid response window
                        response_latencies.append(latency)

        avg_response_latency = np.mean(response_latencies) if response_latencies else 0
        std_response_latency = np.std(response_latencies) if len(response_latencies) > 1 else 0
        max_response_latency = np.max(response_latencies) if response_latencies else 0

        # Total speaking duration
        if 'start_time' in df_part.columns and 'stop_time' in df_part.columns:
            speaking_durations = (df_part['stop_time'] - df_part['start_time']).clip(lower=0)
            total_speak_dur = speaking_durations.sum()
            avg_speak_dur = speaking_durations.mean()
        else:
            total_speak_dur = avg_speak_dur = 0

        features = np.array([
            n_turns,                # Number of speaking turns
            n_words,                # Total words
            lexical_diversity,      # Type-Token Ratio
            avg_words_per_turn,     # Average words per response
            first_person_rate,      # First-person pronoun rate
            negative_rate,          # Negative affect words
            positive_rate,          # Positive affect words
            sentiment_ratio,        # Positive/negative ratio
            filler_rate,            # Filler words (hesitation)
            avg_sent_length,        # Average sentence length
            sent_length_var,        # Variability in sentence length
            avg_response_latency,   # How long to respond
            std_response_latency,   # Variability in response time
            max_response_latency,   # Longest pause before responding
            total_speak_dur,        # Total speaking time
            avg_speak_dur,          # Average speaking turn duration
            n_words / max(total_speak_dur + 1, 1),  # Speech rate (words/sec)
            len(sentences),         # Number of complete sentences
            unique_words,           # Vocabulary richness (absolute)
            n_turns / max(len(df_ellie) + 1, 1),    # Participant/Ellie turn ratio
        ])
        return features
    except Exception as e:
        return np.zeros(20)

linguistic_feats = []
for _, row in df_labels.iterrows():
    pid = int(row['id'])
    feats = extract_linguistic_features(pid, RAW_DIR)
    linguistic_feats.append(feats)

X_linguistic = np.array(linguistic_feats)
X_linguistic = np.nan_to_num(X_linguistic, nan=0.0, posinf=0.0, neginf=0.0)
print(f"Linguistic features shape: {X_linguistic.shape}")


## Feature 4: Gender as Binary Feature


In [ ]:
gender_map = {'male': 0, 'female': 1, 'm': 0, 'f': 1, '0': 0, '1': 1}
gender_feat = df_labels['gender'].astype(str).str.lower().map(gender_map).fillna(0.5).values.reshape(-1, 1)
print(f"Gender feature shape: {gender_feat.shape}")

# Label
y_all = df_labels['label'].values.astype(int)
print(f"\nFull label distribution: {np.unique(y_all, return_counts=True)}")


## Cross-Validation Evaluation: 5-Fold StratifiedCV


In [ ]:
print("\n" + "="*65)
print("5-FOLD STRATIFIED CV — Individual Feature Sets")
print("="*65)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

feature_sets = {
    'BERT_only':        X_bert,
    'Prosodic_only':    X_prosodic,
    'Linguistic_only':  X_linguistic,
    'BERT+Prosodic':    np.hstack([X_bert, X_prosodic]),
    'BERT+Linguistic':  np.hstack([X_bert, X_linguistic]),
    'All_Features':     np.hstack([X_bert, X_prosodic, X_linguistic, gender_feat]),
    'Audio_only':       np.hstack([X_prosodic, X_linguistic]),
}

def evaluate_feature_set(X, y, skf, model, use_smote=True, pca_var=None):
    """Evaluate a feature set with cross-validation."""
    y_oof = np.zeros(len(y))

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        # Robust scaling (handles outliers better)
        scaler = RobustScaler()
        X_tr  = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)

        # PCA if specified
        if pca_var and X_tr.shape[1] > 20:
            pca = PCA(n_components=pca_var, random_state=RANDOM_SEED)
            X_tr  = pca.fit_transform(X_tr)
            X_val = pca.transform(X_val)

        # SMOTE
        if use_smote:
            k = min(3, y_tr.sum() - 1)
            if k >= 1:
                sm = SMOTE(random_state=RANDOM_SEED, k_neighbors=k)
                X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

        # Clone and fit model
        import sklearn.base
        m = sklearn.base.clone(model)
        m.fit(X_tr, y_tr)

        try:
            y_oof[val_idx] = m.predict_proba(X_val)[:, 1]
        except:
            y_oof[val_idx] = m.predict(X_val).astype(float)

    # Tune threshold
    best_f1, best_thr = 0.0, 0.5
    for thr in np.arange(0.25, 0.76, 0.01):
        preds = (y_oof >= thr).astype(int)
        f1 = f1_score(y, preds, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr

    auc = roc_auc_score(y, y_oof)
    return best_f1, best_thr, auc, y_oof

# Models to test
test_models = {
    'LR_C01':    LogisticRegression(C=0.1,  class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED),
    'LR_C05':    LogisticRegression(C=0.5,  class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED),
    'LR_C1':     LogisticRegression(C=1.0,  class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED),
    'SVM_C1':    SVC(C=1.0, kernel='rbf',  probability=True, class_weight='balanced', random_state=RANDOM_SEED),
    'SVM_C5':    SVC(C=5.0, kernel='rbf',  probability=True, class_weight='balanced', random_state=RANDOM_SEED),
    'SVM_lin':   SVC(C=1.0, kernel='linear', probability=True, class_weight='balanced', random_state=RANDOM_SEED),
    'GBM':       GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=RANDOM_SEED),
    'XGB':       xgb.XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05,
                                    scale_pos_weight=2.4, random_state=RANDOM_SEED,
                                    eval_metric='logloss', use_label_encoder=False),
    'LGBM':      lgb.LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.05,
                                     class_weight='balanced', random_state=RANDOM_SEED, verbose=-1),
}

all_cv_results = {}
all_oof_probs  = {}

# Focus on BERT+Linguistic and All_Features (best candidates)
for feat_name in ['BERT_only', 'BERT+Linguistic', 'BERT+Prosodic', 'All_Features']:
    X_feat = feature_sets[feat_name]
    print(f"\n[{feat_name}] shape={X_feat.shape}")

    for model_name, model in test_models.items():
        t0 = time.time()
        f1, thr, auc, oof = evaluate_feature_set(
            X_feat, y_all, skf, model,
            use_smote=True,
            pca_var=0.95 if X_feat.shape[1] > 100 else None
        )
        key = f"{feat_name}|{model_name}"
        all_cv_results[key] = {'F1': f1, 'Thr': thr, 'AUC': auc, 'elapsed': time.time()-t0}
        all_oof_probs[key] = oof
        print(f"  {model_name:<12}: F1={f1:.4f} (thr={thr:.2f}) | AUC={auc:.4f} | {time.time()-t0:.0f}s")


## Results Summary & Best Combinations


In [ ]:
print("\n" + "="*70)
print("TOP-15 COMBINATIONS BY CV MACRO F1")
print("="*70)

df_results = pd.DataFrame(all_cv_results).T
df_results = df_results.sort_values('F1', ascending=False)
print(df_results.head(15).to_string())

best_key = df_results.index[0]
best_f1 = df_results.iloc[0]['F1']
print(f"\n>>> BEST COMBO: {best_key} | CV Macro F1 = {best_f1:.4f}")


## Ensemble: Top-K Weighted Soft Voting


In [ ]:
print("\n" + "="*65)
print("ENSEMBLE — Weighted Soft Voting (Top-K by CV F1)")
print("="*65)

sorted_keys = df_results.index.tolist()
for k_top in [3, 5, 7, 10]:
    top_keys = sorted_keys[:k_top]
    weights  = np.array([all_cv_results[k]['F1'] for k in top_keys])
    weights  = weights / weights.sum()

    oof_ensemble = np.average(
        np.column_stack([all_oof_probs[k] for k in top_keys]),
        axis=1, weights=weights
    )

    best_f1_e, best_thr_e = 0.0, 0.5
    for thr in np.arange(0.25, 0.76, 0.01):
        preds = (oof_ensemble >= thr).astype(int)
        f1 = f1_score(y_all, preds, average='macro', zero_division=0)
        if f1 > best_f1_e:
            best_f1_e, best_thr_e = f1, thr

    auc_e = roc_auc_score(y_all, oof_ensemble)
    print(f"  Top-{k_top:<3} ensemble: F1={best_f1_e:.4f} (thr={best_thr_e:.2f}) | AUC={auc_e:.4f}")

    if k_top == 5:
        best_ensemble_key = f"Top-{k_top}"
        best_ensemble_f1  = best_f1_e
        best_ensemble_thr = best_thr_e
        best_ensemble_oof = oof_ensemble


## Official 80/20 Test Evaluation


In [ ]:
print("\n" + "="*65)
print("OFFICIAL 80/20 STRATIFIED TEST EVALUATION")
print("="*65)

from sklearn.model_selection import train_test_split

idx_all = np.arange(len(y_all))
train_idx, test_idx = train_test_split(
    idx_all, test_size=0.2, random_state=RANDOM_SEED, stratify=y_all
)

y_train = y_all[train_idx]
y_test  = y_all[test_idx]
print(f"Train: {len(train_idx)} | Test: {len(test_idx)}")
print(f"Train labels: {np.unique(y_train, return_counts=True)}")
print(f"Test labels : {np.unique(y_test, return_counts=True)}")

# Best feature set
best_feat_name = best_key.split('|')[0]
best_model_name = best_key.split('|')[1]
X_best = feature_sets[best_feat_name]

X_tr = X_best[train_idx]
X_te = X_best[test_idx]

# Scale
scaler = RobustScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# PCA
if X_tr_s.shape[1] > 100:
    pca = PCA(n_components=0.95, random_state=RANDOM_SEED)
    X_tr_s = pca.fit_transform(X_tr_s)
    X_te_s = pca.transform(X_te_s)
    print(f"After PCA: {X_tr_s.shape}")

# SMOTE
k = min(3, y_train.sum() - 1)
sm = SMOTE(random_state=RANDOM_SEED, k_neighbors=k)
X_tr_res, y_tr_res = sm.fit_resample(X_tr_s, y_train)

# Train best models
best_thr_cv = all_cv_results[best_key]['Thr']
test_probs_all = {}
print("\nTest results for top-10 CV models:")

import sklearn.base
for key in sorted_keys[:10]:
    feat_name = key.split('|')[0]
    model_name = key.split('|')[1]
    X_f = feature_sets[feat_name]

    X_tr_f = X_f[train_idx]
    X_te_f = X_f[test_idx]

    sc = RobustScaler()
    X_tr_f_s = sc.fit_transform(X_tr_f)
    X_te_f_s = sc.transform(X_te_f)

    if X_tr_f_s.shape[1] > 100:
        pca_f = PCA(n_components=0.95, random_state=RANDOM_SEED)
        X_tr_f_s = pca_f.fit_transform(X_tr_f_s)
        X_te_f_s = pca_f.transform(X_te_f_s)

    k_f = min(3, y_train.sum() - 1)
    sm_f = SMOTE(random_state=RANDOM_SEED, k_neighbors=k_f)
    X_tr_f_res, y_tr_f_res = sm_f.fit_resample(X_tr_f_s, y_train)

    model = sklearn.base.clone(test_models[model_name])
    model.fit(X_tr_f_res, y_tr_f_res)

    try:
        probs = model.predict_proba(X_te_f_s)[:, 1]
    except:
        probs = model.predict(X_te_f_s).astype(float)

    thr = all_cv_results[key]['Thr']
    preds = (probs >= thr).astype(int)
    f1  = f1_score(y_test, preds, average='macro', zero_division=0)
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    print(f"  {key:<40}: Test F1={f1:.4f} | Acc={acc:.4f} | AUC={auc:.4f}")
    test_probs_all[key] = probs

# Ensemble on test
print("\n--- Test Ensemble ---")
for k_top in [3, 5, 7]:
    top_keys = sorted_keys[:k_top]
    w = np.array([all_cv_results[k]['F1'] for k in top_keys if k in test_probs_all])
    available_keys = [k for k in top_keys if k in test_probs_all]
    if not available_keys:
        continue
    w = w[:len(available_keys)]
    w = w / w.sum()
    ens_probs = np.average(np.column_stack([test_probs_all[k] for k in available_keys]),
                            axis=1, weights=w)
    best_f1_t, best_thr_t = 0.0, 0.5
    for thr in np.arange(0.25, 0.76, 0.01):
        preds = (ens_probs >= thr).astype(int)
        f1 = f1_score(y_test, preds, average='macro', zero_division=0)
        if f1 > best_f1_t:
            best_f1_t, best_thr_t = f1, thr
    ens_auc = roc_auc_score(y_test, ens_probs)
    ens_preds = (ens_probs >= best_thr_t).astype(int)
    ens_acc = accuracy_score(y_test, ens_preds)
    print(f"  Top-{k_top} Weighted Ensemble: F1={best_f1_t:.4f} (thr={best_thr_t:.2f}) | Acc={ens_acc:.4f} | AUC={ens_auc:.4f}")


## Final Summary


In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY v49 — BERT + PROSODIC + LINGUISTIC (189 Participants)")
print("="*70)
print(f"Best CV Configuration: {best_key}")
print(f"Best CV Macro F1     : {best_f1:.4f}")

# Best overall test result
all_test_f1 = {}
for key, probs in test_probs_all.items():
    thr = all_cv_results[key]['Thr']
    preds = (probs >= thr).astype(int)
    f1 = f1_score(y_test, preds, average='macro', zero_division=0)
    all_test_f1[key] = f1

best_test_key = max(all_test_f1, key=all_test_f1.get)
best_test_f1  = all_test_f1[best_test_key]

print(f"\nBest TEST Configuration: {best_test_key}")
print(f"Best TEST Macro F1     : {best_test_f1:.4f}")

# Classification report
best_probs_final = test_probs_all[best_test_key]
best_thr_final   = all_cv_results[best_test_key]['Thr']
best_preds_final = (best_probs_final >= best_thr_final).astype(int)
print(f"\nClassification Report ({best_test_key}):")
print(classification_report(y_test, best_preds_final,
                             target_names=['Non-Depressed', 'Depressed'], zero_division=0))

# Save
df_final_cv = pd.DataFrame(all_cv_results).T.sort_values('F1', ascending=False)
df_final_cv.to_csv(os.path.join(RESULTS_DIR, "metrics", "v49_cv_results.csv"))
print(f"Results saved to: {RESULTS_DIR}")

print()
if best_test_f1 >= 0.70:
    print(f"🎯 TARGET ACHIEVED! Test Macro F1 = {best_test_f1:.4f} >= 0.70 ✅")
elif best_f1 >= 0.70:
    print(f"🎯 CV TARGET ACHIEVED! CV Macro F1 = {best_f1:.4f} >= 0.70 ✅")
    print(f"   (Test F1 = {best_test_f1:.4f} — slight generalization gap)")
else:
    print(f"⚠️  Target NOT yet achieved.")
    print(f"   Best CV F1  : {best_f1:.4f} | Gap from 0.70: {0.70-best_f1:.4f}")
    print(f"   Best Test F1: {best_test_f1:.4f} | Gap from 0.70: {0.70-best_test_f1:.4f}")